# MetroPT-3 data exploration

Run `uv run metropt download` and `uv run metropt prepare` from the repository first. Reusable inspection logic lives in `src/anomaly_detection/eda.py`.

Source: [UCI MetroPT-3](https://archive.ics.uci.edu/dataset/791/metropt+3+dataset), CC BY 4.0. Failure intervals are weak annotations; unreported data are unknown.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display
from anomaly_detection.config import load_config
from anomaly_detection.data import CONTINUOUS_COLUMNS, DIGITAL_COLUMNS
root = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
config = load_config(root / "configs/experiments.yaml")
observations = pd.read_parquet(root / config["data"]["processed_path"])
observations.head()

## Cadence, gaps, and chronology
The published CSV has an approximately ten-second native cadence. Causal minute snapshots reduce CPU work; no future value or interpolation fills a missing timestamp. Inspect the measured report instead of assuming perfect sampling.

In [ ]:
quality = json.loads((root / "reports/data_quality.json").read_text())
quality

In [ ]:
observations[CONTINUOUS_COLUMNS].describe().T

In [ ]:
{column: observations[column].value_counts().to_dict() for column in DIGITAL_COLUMNS}

In [ ]:
pd.read_csv(root / "reports/failure_timeline.csv")

## Sensors, failure reports, and operating states
Shaded bands show reported failures, not exhaustive sensor labels. Motor-current bands describe operating behavior and are not normal/failure targets. Explore February for modeling choices; use the frozen validation protocol for selection and the final test once for assessment.

In [ ]:
for figure in ["sensor_timeline.png", "sensor_distributions.png", "digital_signals.png", "sensor_correlations.png", "operating_states.png"]:
    display(Image(filename=str(root / "reports" / figure)))

## Fixed protocol
February 1–22: normal-only fitting. February 23–29: normal reconstruction development. March 1–June 14: validation events F1–F3. June 15–September 1: final event F4. Feature and sequence histories reset at every partition. See `docs/data.md` for source discrepancies and limits.